<img align="right" src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/PyBMB_logo.png" width="150" height="150" />

# Project: CodeBMB: Computational Literacy for Biochemistry and Molecular Biology Education
## Notebook 004T: Linear Regression — Faculty Template

**Purpose:**
This notebook guides students through a complete linear regression
workflow using the Bradford protein assay as an example. Faculty
can adapt this template to any quantitative assay that produces a
linear standard curve by modifying the variables at each
🔧 TEMPLATE MODIFICATION POINT.

In this lesson students will learn to:
- Load experimental data and inspect it using pandas
- Calculate means and standard deviations across replicates
- Build a standard curve with error bars using matplotlib
- Perform linear regression using `scipy.stats.linregress`
- Interpret slope, intercept, and R² in a biological context
- Account for sample dilution when calculating true concentrations
- Apply the complete workflow to any linear quantitative assay

> 🔧 **This notebook is a faculty template.**
> The Bradford assay is used as the working example throughout.
> Every place where you adapt it to your own assay is marked with 🔧.

**Complete List of Template Modification Points:**

| Section | Variable | Current Value | What to Change |
|---|---|---|---|
| 3a | `assay_name` | `'Bradford Protein Assay'` | Your assay name |
| 3a | `wavelength` | `595` | Your measurement wavelength (nm) |
| 3a | `conc_units` | `'mg/mL'` | Your concentration units |
| 3a | `concentration_col` | `'Concentration_mg_per_mL'` | Your CSV column name |
| 3a | `replicate_cols` | `['Rep1','Rep2','Rep3']` | Your replicate column names |
| 3b | Data source | GitHub CSV | Your data (GitHub / dict / CSV) |
| 5.5 | `sample_names` | `["CFE","FT","E1","E2"]` | Your sample names |
| 5.5 | `sample_descriptions` | Purification fractions | Your sample descriptions |
| 5.5 | `sample_abs` | `[0.492, 0.427, 0.284, 0.199]` | Your absorbance readings |
| 5.5 | `dilution_factors` | `[10, 5, 2, 1]` | Your dilution factors (1 = undiluted) |
| 6 | Interpretation questions | Bradford/purification | Your experiment context |

**Input Data:**
* **Description:** Bradford assay standard curve — BSA standards
  with three replicates at six concentrations (0.00–0.50 mg/mL)
  measured at 595 nm.
* **Source:** `bsa_std_curve.csv` — CodeBMB April 2025 workshop dataset
* **Retrieved On:** 2025-04-26
* **Access:** Downloaded automatically from CodeBMB GitHub repository

**Libraries:**
* `pandas` — loading and organizing data in DataFrames
* `numpy` — numerical operations and generating best-fit line points
* `matplotlib` — creating and customizing plots
* `scipy.stats` — performing linear regression with `linregress`

**Status with Date:** April 2025 — Faculty template version

**License**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100"/>
CC BY-NC-SA — reusers may distribute, remix, adapt, and build upon
the material for noncommercial purposes only, with attribution,
under identical terms.

---
**Authorship:** Zinedine Sehili, Paul Craig, Wally Novak

**Acknowledgements:** This workshop is supported by NSF IUSE 2518733.

**Contact Info:** codingBMB@gmail.com

# 0. How to Run This Notebook

To run a cell, click the ▶ play button on the left side of the cell.

![run button image](https://github.com/wallynovak/biochemistry_seq_analysis/blob/main/images/run.png?raw=1)

A cell is still running if you see a stop button with a moving circle.
A completed cell shows a number in brackets (e.g. [1]) and a checkmark.

**Please ensure every cell finishes before running the next one.**

> 🔧 **For faculty adapting this template:** Before sharing with
> students, go to **Runtime → Restart and clear all outputs** to
> reset the notebook to a clean state.

# 1. Environment Setup and Libraries

## Libraries Used in This Notebook

| Library | Purpose |
|---|---|
| `pandas` | Loading the CSV file and organizing data into a table |
| `numpy` | Generating points for the best-fit line |
| `matplotlib` | Creating and customizing plots |
| `scipy.stats` | Performing linear regression with `linregress` |

Run this cell first — everything below depends on it.

In [ ]:
# Run this cell to import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import urllib.request
import os

print("✅ All libraries imported successfully!")

# 2. Theory Block

> 🔧 **FACULTY NOTE — Theory Block**
> The explanation below uses the Bradford assay as the example.
> Replace the highlighted sections with a description of your
> own assay and its biochemical basis. The Beer-Lambert Law,
> linear regression, and dilution correction sections apply
> universally and do not need to change.

---

## The Bradford Protein Assay

The **Bradford assay** is one of the most widely used methods in
biochemistry for measuring protein concentration. It relies on a dye —
**Coomassie Brilliant Blue G-250** — that shifts from reddish-brown
to blue when it binds to protein, particularly at arginine residues.

We measure this shift at **595 nm**:

> **More protein → more dye binding → deeper blue → higher absorbance**

---

## Why Linear Regression?

The Bradford assay follows **Beer-Lambert Law**, which tells us
absorbance is proportional to concentration:

$$A = \varepsilon \cdot l \cdot c$$

Since molar absorptivity (ε) and path length (l) are constants,
this simplifies to a straight line — just like y = mx + b:

$$A_{595} = slope \times [Protein] + intercept$$

| Variable | Role | In our experiment |
|---|---|---|
| $y$ | What we measure | Absorbance |
| $x$ | What we control | Protein concentration |
| $m$ | Slope | Change in absorbance per unit concentration |
| $b$ | Y-intercept | Absorbance when concentration = 0 |

**Linear regression** finds the best values of slope and intercept
to describe our standard curve. We use `scipy.stats.linregress`.

---

## What Is R²?

| R² Value | What it means |
|---|---|
| R² = 1.00 | Perfect fit — every point lies exactly on the line |
| R² ≥ 0.99 | Excellent — expected for a good Bradford assay |
| R² ≥ 0.95 | Acceptable |
| R² < 0.90 | Poor — check for pipetting errors |

---

## Accounting for Sample Dilution

In many experiments, samples must be **diluted** before measurement
to bring their absorbance into the range of the standard curve.
The concentration calculated from the regression equation is the
**measured concentration** — the concentration in the diluted sample.
To get the **true concentration** in the original sample we multiply
by the dilution factor:

$$[Protein]_{true} = [Protein]_{measured} \times dilution\ factor$$

For example, if a sample was diluted 1:10 (dilution factor = 10)
and the measured concentration is 0.40 mg/mL, the true concentration
in the original sample is 4.0 mg/mL.

> If no dilution was performed, the dilution factor = 1 and
> measured concentration = true concentration.

---

## Solving for Unknown Concentrations

Rearranging our regression equation to solve for concentration:

$$[Protein] = \frac{A_{595} - intercept}{slope}$$

This is the practical payoff of this notebook — used in Section 5.5
to find true concentrations in unknown samples.

# 3. Data Acquisition

## 3a. Define Your Assay Parameters

Before loading any data, define the key parameters for your assay.
These variables are used throughout the notebook — in axis labels,
column references, print statements, and calculations — so you only
need to change them once here.

> 🔧 **TEMPLATE MODIFICATION POINT**
> Change these five variables to match your assay.
> The rest of the notebook will update automatically.

In [ ]:
# ─────────────────────────────────────────────────────────
# 🔧 TEMPLATE MODIFICATION POINT — Change these five variables
# ─────────────────────────────────────────────────────────

assay_name       = 'Bradford Protein Assay'   # 🔧 Your assay name
wavelength       = 595                         # 🔧 Measurement wavelength (nm)
conc_units       = 'mg/mL'                    # 🔧 Concentration units
concentration_col = 'Concentration_mg_per_mL' # 🔧 First column name in your CSV
replicate_cols   = ['Rep1', 'Rep2', 'Rep3']   # 🔧 Replicate column names

# ─────────────────────────────────────────────────────────
# No changes needed below this line
# ─────────────────────────────────────────────────────────
print(f"✅ Parameters set:")
print(f"   Assay:       {assay_name}")
print(f"   Wavelength:  {wavelength} nm")
print(f"   Units:       {conc_units}")
print(f"   Conc column: {concentration_col}")
print(f"   Replicates:  {replicate_cols}")

## 3b. Load Your Standard Curve Data

There are **three ways** to load your standard curve data into
this notebook. **Run only ONE of the three options below.**

| Option | Best for | Account needed |
|---|---|---|
| **A — GitHub download** | Workshop participants; shared datasets | No |
| **B — Manual entry** | Small datasets; typing data directly | No |
| **C — CSV upload** | Your own data file from your computer | No |

---

> 🔧 **FACULTY NOTE — Data Format**
> Whichever option you choose, your data must have:
> - One column of **concentration values** (your x-axis)
> - One or more columns of **replicate measurements** (your y-axis readings)
> - Column names that **exactly match** `concentration_col`
>   and `replicate_cols` defined in Section 3a

---

### ▶ OPTION A — Download from GitHub (default for workshop)

In [ ]:
# OPTION A — Download data from GitHub
# Run this cell if your data file is stored in a GitHub repository

# 🔧 Replace this URL with the raw GitHub URL of your CSV file
data_url = 'https://raw.githubusercontent.com/codeBMB/April26_Rutgers-RCSB/main/data/bsa_std_curve.csv'

os.makedirs('data', exist_ok=True)
urllib.request.urlretrieve(data_url, 'data/std_curve.csv')

df = pd.read_csv('data/std_curve.csv')

print("✅ Data downloaded from GitHub successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

### ▶ OPTION B — Enter Data Manually

Use this option if you have a small dataset and prefer to
type it directly into the notebook.

The data must be organized as a Python dictionary:
- One key matching `concentration_col` with a list of concentration values
- Additional keys matching each name in `replicate_cols` with lists
  of absorbance readings

> 🔧 **TEMPLATE MODIFICATION POINT**
> Replace the values below with your own standard curve data.
> Every list must have the same number of values.

In [ ]:
# OPTION B — Manual data entry
# Replace the values below with your own standard curve data
# Every list must be the same length

# 🔧 TEMPLATE MODIFICATION POINT — replace all values below
std_curve_data = {
    concentration_col: [0.00, 0.10, 0.20, 0.30, 0.40, 0.50],  # 🔧 concentrations
    'Rep1':            [-0.004, 0.118, 0.238, 0.356, 0.478, 0.596],  # 🔧 replicate 1
    'Rep2':            [ 0.001, 0.121, 0.242, 0.361, 0.482, 0.601],  # 🔧 replicate 2
    'Rep3':            [ 0.003, 0.124, 0.245, 0.364, 0.485, 0.604],  # 🔧 replicate 3
}

df = pd.DataFrame(std_curve_data)

print("✅ DataFrame created from manual data entry!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

### ▶ OPTION C — Upload Your Own CSV File

Use this option if you have collected your own data and saved
it as a CSV file on your computer.

---

#### What Your CSV Must Look Like

Your CSV file must have the following structure:

| Concentration_mg_per_mL | Rep1 | Rep2 | Rep3 |
|---|---|---|---|
| 0.00 | -0.004 | 0.001 | 0.003 |
| 0.10 | 0.118 | 0.121 | 0.124 |
| 0.20 | 0.238 | 0.242 | 0.245 |
| ... | ... | ... | ... |

**Rules:**
- The **first row** must be a header row with column names
- The **first column** must contain your concentration values
- The **remaining columns** must contain replicate measurements
- Column names must **exactly match** `concentration_col` and
  `replicate_cols` defined in Section 3a
- Values must be **numbers only** — no units in the data cells
- Save the file as `.csv` (Comma Separated Values)

> **In Excel or Google Sheets:**
> File → Download → Comma Separated Values (.csv)

---

#### How to Upload to Colab

There are two ways:

**Method 1 — Drag and drop:**
1. Click the **folder icon (📁)** in the left panel
2. Drag your CSV file from your computer into the panel

**Method 2 — Upload dialog (used in the code below):**
1. Run the cell below
2. Click **Choose Files** in the dialog that appears
3. Navigate to and select your CSV file
4. Wait for the upload to complete

> ⚠️ **Important:** Uploaded files are temporary in Colab.
> If your session restarts, you will need to upload again.
> Consider saving a copy to Google Drive using the code
> in Notebook 001, Section 9.

> 🔧 **TEMPLATE MODIFICATION POINT**
> After uploading, update the `uploaded_filename` variable
> below to match the exact name of your uploaded file.

In [ ]:
# OPTION C — Upload your own CSV file from your computer

from google.colab import files

print("A file chooser dialog will appear below.")
print("Select your CSV file and wait for the upload to complete.")
print()

# This opens the file upload dialog
uploaded = files.upload()

# 🔧 If your file has a different name, update it here
# The filename is shown after upload completes
uploaded_filename = list(uploaded.keys())[0]

print(f"\n✅ File uploaded: {uploaded_filename}")

# Read the uploaded file into a DataFrame
df = pd.read_csv(uploaded_filename)

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("Column names found in your file:")
print(list(df.columns))
print()
print("⚠️  Check that column names match concentration_col and")
print(f"   replicate_cols defined in Section 3a:")
print(f"   concentration_col = '{concentration_col}'")
print(f"   replicate_cols    = {replicate_cols}")
df.head()

## 3c. Verifying Your Data Loaded Correctly

Regardless of which option you used above, run this cell to
confirm that your data loaded correctly before continuing.

Check that:
- The **shape** matches the number of rows and columns you expect
- The **column names** match what you defined in Section 3a
- The **data types** are numeric (`float64` or `int64`) for all columns
- The **head** shows sensible values — no obvious errors

In [ ]:
# Verify your data — run this regardless of which option you used

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("Column names:")
print(list(df.columns))
print()
print("Data types:")
print(df.dtypes)
print()
print("First rows:")
df.head()

# 4. Data Inspection and Cleaning

Before any analysis, we clean the data using `dropna()` to remove
any rows containing missing values (NaN). With a carefully prepared
dataset we don't expect any — but this step is good practice for
any real experimental dataset where instruments sometimes fail to
record a value.

We also use `.copy()` to avoid a pandas `SettingWithCopyWarning`
when we add new columns later.

`clean_data_df.head()` confirms all rows loaded correctly.
Check that you can see the full range of concentrations and
three replicate absorbance readings for each.

In [ ]:
# Clean the data — remove any rows with missing values
clean_data_df = df.dropna().copy()

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning:  {len(clean_data_df)}")
print()
clean_data_df.head()

# 5. Data Analysis

With our data loaded and cleaned, we work through five steps:

1. Calculate **mean absorbance** and **standard deviation** across replicates
2. **Plot** the standard curve to visually inspect the data
3. Add **error bars** to show measurement variability
4. Perform **linear regression** and evaluate the fit with R²
5. Use the regression equation to calculate **true concentrations**
   in unknown samples, correcting for dilution

Run each cell in order and read the output before moving on.

## 5.1 Calculating the Average and Standard Deviation

Each concentration point was measured multiple times (one reading
per replicate column). We collapse these into:
- **Average** — the central value we will plot and fit
- **Standard deviation** — the spread we will show as error bars

The key detail here is `axis=1`. In pandas:
- `axis=0` calculates *down* the rows (one result per column)
- `axis=1` calculates *across* the columns (one result per row)

Since our replicates are in separate columns and we want one
average **per row** (per concentration), we use `axis=1`.

> 🔧 This cell uses `replicate_cols` defined in Section 3a.
> If you have more or fewer replicates, update `replicate_cols`
> there and this cell will work automatically.

In [ ]:
# 5.1 Calculate average and standard deviation across replicates
# Uses replicate_cols defined in Section 3a — no changes needed here

# Average across replicate columns for each row
clean_data_df['average'] = clean_data_df[replicate_cols].mean(axis=1)

# Standard deviation across replicate columns for each row
clean_data_df['std_dev'] = clean_data_df[replicate_cols].std(axis=1)

# Display the updated DataFrame
clean_data_df.head()

## 📝 Exercise 5.1 — Exploring the Standard Curve Data

Look at the DataFrame output above and answer the following:

1. Does the absorbance at the lowest concentration make sense?
   What would you expect from a well-blanked instrument?

2. In the code cell below, `print()` the average absorbance at
   the highest concentration point.
   *Hint: use `clean_data_df['average'].iloc[-1]`*

3. Add a new column called `'CV_percent'` (coefficient of variation):

$$CV\% = \frac{standard\ deviation}{average} \times 100$$

*Hint: follow the same pattern used to calculate `'average'`
and `'std_dev'` in the cell above.*

In [ ]:
# 1. Print the average absorbance at the highest concentration


# 2. Calculate CV_percent
#    CV_percent = (std_dev / average) * 100


# Display the updated DataFrame
clean_data_df.head()

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

~~~python
# 1. Print the average at the highest concentration
print(clean_data_df['average'].iloc[-1])

# 2. Calculate CV%
clean_data_df['CV_percent'] = (
    clean_data_df['std_dev'] / clean_data_df['average']
) * 100

clean_data_df.head()
~~~

> **Note:** CV% at the lowest concentration may be very large or
> undefined if the average is close to zero — this is expected.

</details>

## 5.2 Plotting the Standard Curve

Before fitting any model, always **look at your data first.**
A scatter plot immediately shows us:
- Is the relationship linear?
- Does the absorbance start near zero at zero concentration?
- Are there any obvious outliers?

> 🔧 This cell uses `concentration_col`, `wavelength`, `conc_units`,
> and `assay_name` defined in Section 3a. No changes needed.

In [ ]:
# 5.2 Plot the standard curve
# Axis labels and title update automatically from Section 3a variables

plt.figure(figsize=(8, 6))
plt.plot(
    clean_data_df[concentration_col],
    clean_data_df['average'],
    'o',
    label='Average Absorbance'
)
plt.xlabel(f'Concentration ({conc_units})')
plt.ylabel(f'Absorbance at {wavelength} nm')
plt.title(f'{assay_name} Standard Curve')
plt.grid(True)
plt.legend()
plt.show()

## 5.3 Adding Error Bars

Our assay used multiple replicates per concentration, giving us
information about the **consistency** of our measurements.
Error bars display the standard deviation at each point.

### Step 1 — Reading the Documentation

Run the cell below to see the full documentation for
`plt.errorbar` directly in Colab.
The key parameters for our use are `yerr`, `fmt`, and `capsize`.

In [ ]:
# Access matplotlib errorbar documentation in Colab
# Focus on the Parameters section — especially yerr, fmt, and capsize
help(plt.errorbar)

The documentation shows that `errorbar` accepts many optional
parameters. The three we use:
- `yerr` — the values that set the error bar size (our std_dev)
- `fmt` — controls what the data points look like (`'o'` = circles)
- `capsize` — the small horizontal caps at the top and bottom

### Step 2 — Compare With and Without Error Bars

In [ ]:
# Compare plt.plot() and plt.errorbar() side by side

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: no error bars
axes[0].plot(
    clean_data_df[concentration_col],
    clean_data_df['average'],
    'o', color='steelblue', label='Average'
)
axes[0].set_xlabel(f'Concentration ({conc_units})')
axes[0].set_ylabel(f'Absorbance at {wavelength} nm')
axes[0].set_title('Without Error Bars')
axes[0].grid(True)
axes[0].legend()

# Right: with error bars
axes[1].errorbar(
    clean_data_df[concentration_col],
    clean_data_df['average'],
    yerr=clean_data_df['std_dev'],
    fmt='o', capsize=4, color='steelblue',
    label='Average ± Std Dev'
)
axes[1].set_xlabel(f'Concentration ({conc_units})')
axes[1].set_ylabel(f'Absorbance at {wavelength} nm')
axes[1].set_title('With Error Bars')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

Compare the two plots. The error bars show how consistent the
replicates were at each concentration.

A well-prepared standard curve will show **small, tight error bars**
across the full concentration range. If one point has unusually
large bars it may indicate a pipetting error worth investigating.

### Step 3 — Final Error Bar Plot

In [ ]:
# 5.3 Final error bar plot

plt.figure(figsize=(8, 6))
plt.errorbar(
    clean_data_df[concentration_col],
    clean_data_df['average'],
    yerr=clean_data_df['std_dev'],
    fmt='o',
    capsize=4,
    label='Average ± Std Dev'
)
plt.xlabel(f'Concentration ({conc_units})')
plt.ylabel(f'Absorbance at {wavelength} nm')
plt.title(f'{assay_name} Standard Curve with Error Bars')
plt.grid(True)
plt.legend()
plt.show()

## 5.4 Fitting the Line: Linear Regression

Now we fit a straight line using `scipy.stats.linregress`.
Before we run it, let's understand what it returns.

### Step 1 — Reading the Documentation

Official documentation:
[https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html)

Run the cell below to access the documentation in Colab.

In [ ]:
# Access scipy linregress documentation in Colab
# Focus on the Returns section — five named values
help(stats.linregress)

### Step 2 — What Does linregress Return?

According to the documentation, `linregress` returns five values:

| Name | What it means |
|---|---|
| `slope` | Change in absorbance per unit concentration |
| `intercept` | Expected absorbance when concentration = 0 |
| `rvalue` | Correlation coefficient r — **not** R² yet |
| `pvalue` | Statistical significance of the relationship |
| `stderr` | Uncertainty in the slope estimate |

> **Important:** The function returns `rvalue` — not R².
> To get R² you must square it: `r_squared = rvalue**2`

### Step 3 — Explore the Result Object

In [ ]:
# Step 3: Store the full result object and explore it

concentration      = clean_data_df[concentration_col]
average_absorbance = clean_data_df['average']

# Store the full result — explore before unpacking
result = stats.linregress(concentration, average_absorbance)

print("Full result object:")
print(result)
print()
print("Accessing values with dot notation:")
print(f"  slope:     {result.slope}")
print(f"  intercept: {result.intercept}")
print(f"  rvalue:    {result.rvalue}")
print(f"  pvalue:    {result.pvalue}")
print(f"  stderr:    {result.stderr}")
print()
print(f"Type of result: {type(result)}")

### Step 4 — Unpack and Calculate R²

Now we unpack all five values into named variables and
calculate R² by squaring `r_value`.

In [ ]:
# Unpack all five return values into named variables
slope, intercept, r_value, p_value, std_err = stats.linregress(
    concentration, average_absorbance
)

# Calculate R-squared by squaring the correlation coefficient
r_squared = r_value**2

# Print a clean summary
print(f"{assay_name} — Linear Regression Results")
print("=" * 45)
print(f"  Slope:       {slope:.4f}  Abs{wavelength} per {conc_units}")
print(f"  Intercept:   {intercept:.4f}")
print(f"  r value:     {r_value:.4f}")
print(f"  R²:          {r_squared:.4f}")
print(f"  P-value:     {p_value:.2e}")
print(f"  Std Error:   {std_err:.4f}")
print()

# Evaluate fit quality
if r_squared >= 0.99:
    print("Fit quality: Excellent ✅")
elif r_squared >= 0.95:
    print("Fit quality: Acceptable ⚠️")
else:
    print("Fit quality: Check your data ❌")

### Step 5 — Plot the Best-Fit Line

Now we add the regression line to our error bar plot.
The equation and R² appear in the legend.

In [ ]:
# 5.4 Plot with best-fit line

# Generate 100 evenly-spaced points for a smooth line
line_x = np.linspace(concentration.min(), concentration.max(), 100)
line_y = slope * line_x + intercept

plt.figure(figsize=(8, 6))

# Error bar data points
plt.errorbar(
    concentration,
    average_absorbance,
    yerr=clean_data_df['std_dev'],
    fmt='o',
    capsize=4,
    label='Average ± Std Dev'
)

# Best-fit line
plt.plot(
    line_x, line_y,
    color='red',
    label=f'Best-fit: y = {slope:.3f}x + {intercept:.3f}\nR² = {r_squared:.3f}'
)

plt.xlabel(f'Concentration ({conc_units})')
plt.ylabel(f'Absorbance at {wavelength} nm')
plt.title(f'{assay_name} Standard Curve with Best-fit Line')
plt.grid(True)
plt.legend()
plt.show()

## 📝 Exercise 5.4 — Interpreting the Regression Results

In the code cell below:

1. Use `print()` and f-strings to display `slope`, `intercept`,
   `r_squared`, `p_value`, and `std_err` with clear labels.
   *Hint: `print(f"Slope: {slope:.4f}")`*

2. Write an `if/elif/else` statement that prints:
   - **"Excellent fit ✅"** if `r_squared >= 0.99`
   - **"Acceptable fit ⚠️"** if `r_squared >= 0.95`
   - **"Check your data ❌"** if `r_squared < 0.95`

3. In plain language — what does the **slope** value tell you
   about how your assay responds to the analyte?

In [ ]:
# 1. Print all regression statistics with labels
print(f"Slope:      ")   # complete these f-strings
print(f"Intercept:  ")
print(f"R²:         ")
print(f"P-value:    ")
print(f"Std Error:  ")
print()

# 2. Write an if/elif/else statement to evaluate fit quality


<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

~~~python
# 1. Print regression statistics
print(f"Slope:      {slope:.4f} Abs{wavelength} per {conc_units}")
print(f"Intercept:  {intercept:.4f}")
print(f"R²:         {r_squared:.4f}")
print(f"P-value:    {p_value:.2e}")
print(f"Std Error:  {std_err:.4f}")
print()

# 2. Evaluate the fit
if r_squared >= 0.99:
    print("Excellent fit ✅")
elif r_squared >= 0.95:
    print("Acceptable fit ⚠️")
else:
    print("Check your data ❌")
~~~

> **About the slope:** The slope tells you how much the absorbance
> changes per unit of concentration. A steeper slope means the assay
> is more sensitive — small changes in concentration produce larger
> changes in signal.

</details>

## 5.5 Calculating True Concentrations in Unknown Samples

We now apply our regression equation to unknown samples.
In this example the samples come from a **protein purification
experiment** — each fraction was collected at a different stage:

| Sample | Label | Description |
|---|---|---|
| CFE | Cell-Free Extract | All proteins from broken cells |
| FT | Flowthrough | Proteins that did NOT bind to the column |
| E1 | Elution 1 | First fraction where target protein eluted |
| E2 | Elution 2 | Second elution fraction |

Because these samples vary widely in protein concentration, they
were **diluted** before measurement to bring their absorbance
into the range of the standard curve.

We therefore calculate in two steps:

$$[Protein]_{measured} = \frac{Abs - intercept}{slope}$$

$$[Protein]_{true} = [Protein]_{measured} \times dilution\ factor$$

> 🔧 **TEMPLATE MODIFICATION POINT — Section 5.5**
> Replace the sample data in the next cell with your own.
> If no dilution was performed, set all dilution factors to 1.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 🔧 TEMPLATE MODIFICATION POINT
# Replace ALL values below with your own sample data
# ─────────────────────────────────────────────────────────────

# Sample names — short labels for your samples
sample_names = ["CFE", "FT", "E1", "E2"]         # 🔧

# Sample descriptions — longer descriptions for the table
sample_descriptions = [
    "Cell-Free Extract",                           # 🔧
    "Flowthrough",                                 # 🔧
    "Elution 1",                                   # 🔧
    "Elution 2",                                   # 🔧
]

# Measured absorbance values for each sample
# These should be within the range of your standard curve
sample_abs = [0.492, 0.427, 0.284, 0.199]         # 🔧

# Dilution factors for each sample
# Set to 1 if the sample was not diluted before measurement
dilution_factors = [10, 5, 2, 1]                  # 🔧

# ─────────────────────────────────────────────────────────────
# Verify the data looks correct before continuing
# ─────────────────────────────────────────────────────────────
print("Sample data entered:")
print(f"{'Sample':<8} {'Description':<22} {'Absorbance':<12} {'Dilution'}")
print("-" * 55)
for n, d, a, f in zip(sample_names, sample_descriptions,
                       sample_abs, dilution_factors):
    print(f"{n:<8} {d:<22} {a:<12} {f}")

### Approach 1 — Dictionary and For Loop

We first calculate concentrations using a `for` loop —
connecting back to what you learned in **Notebook 002**.
This approach prints a quick readable summary.

In [ ]:
# Approach 1: For loop with dictionary
# Calculates measured and true concentrations for each sample

print(f"Sample  {'Measured':>12}  {'Dilution':>10}  {'True':>12}")
print(f"        {f'({conc_units})':>12}  {'Factor':>10}  {f'({conc_units})':>12}")
print("-" * 52)

sample_dict = dict(zip(sample_names, sample_abs))
dilution_dict = dict(zip(sample_names, dilution_factors))

for name, abs_val in sample_dict.items():
    # Step 1: calculate measured concentration from regression equation
    measured = (abs_val - intercept) / slope
    # Step 2: multiply by dilution factor to get true concentration
    true_conc = measured * dilution_dict[name]
    print(f"{name:<8} {measured:>12.3f}  {dilution_dict[name]:>10}  {true_conc:>12.3f}")

Now let's calculate the same concentrations using a **pandas
DataFrame** — connecting back to **Notebook 003**. This approach
produces a formatted table that is easier to read, share, and export.

Notice that both approaches give exactly the same results.
The DataFrame approach also stores all the intermediate calculations
so you can audit your work.

In [ ]:
# Approach 2: pandas DataFrame
# Produces a formatted table with all intermediate calculations

# Step 1: Create the DataFrame from sample data
sample_df = pd.DataFrame({
    'Sample':          sample_names,
    'Description':     sample_descriptions,
    f'Abs{wavelength}': sample_abs,
    'Dilution_Factor': dilution_factors
})

# Step 2: Calculate measured concentration using regression equation
# This applies the equation to every row at once
sample_df['Measured_Conc'] = (
    (sample_df[f'Abs{wavelength}'] - intercept) / slope
).round(3)

# Step 3: Apply dilution correction
# True concentration = measured concentration × dilution factor
sample_df['True_Conc'] = (
    sample_df['Measured_Conc'] * sample_df['Dilution_Factor']
).round(3)

# Step 4: Rename columns to include units for clarity
sample_df = sample_df.rename(columns={
    'Measured_Conc': f'Measured ({conc_units})',
    'True_Conc':     f'True ({conc_units})'
})

# Display the final table
print(f"\n{assay_name} — Sample Concentrations")
print(f"Regression: y = {slope:.4f}x + {intercept:.4f}  |  R² = {r_squared:.4f}")
print()
sample_df

## 📝 Exercise 5.5 — Adding a New Unknown Sample

A colleague just handed you one more sample to analyze:

- **Wash fraction (W1):** Abs = 0.357, diluted 1:3 (dilution factor = 3)

In the code cell below:

1. Add `"W1"` to `sample_names`, `"Wash fraction"` to
   `sample_descriptions`, `0.357` to `sample_abs`, and
   `3` to `dilution_factors` in the 🔧 code cell above
   (Section 5.5 template cell), then re-run both approach cells.

2. Look at the final table. Does W1's true concentration
   make biological sense given where it falls in the purification?

3. Which sample has the highest **true** concentration?
   Is it the same as the highest **measured** concentration?
   Why might they differ?

In [ ]:
# After adding W1 to the template cell above and re-running,
# answer the questions below as comments or print statements

# 3. Which sample has the highest true concentration?
print("Sample with highest TRUE concentration:")
# your code here

# Which has the highest MEASURED concentration?
print("Sample with highest MEASURED concentration:")
# your code here

<details>
<summary>🔍 CLICK TO SEE ANSWER</summary>

~~~python
# After adding W1 to the template cell and re-running:

# Highest true concentration
max_true_idx = sample_df[f'True ({conc_units})'].idxmax()
print(f"Highest true concentration: {sample_df.loc[max_true_idx, 'Sample']}")
print(f"  True conc: {sample_df.loc[max_true_idx, f'True ({conc_units})']}")

# Highest measured concentration
max_meas_idx = sample_df[f'Measured ({conc_units})'].idxmax()
print(f"\nHighest measured concentration: {sample_df.loc[max_meas_idx, 'Sample']}")
print(f"  Measured conc: {sample_df.loc[max_meas_idx, f'Measured ({conc_units})']}")
~~~

> **Thinking about the biology:** The CFE (Cell-Free Extract)
> was diluted the most (10×) before measurement. So even though
> its *measured* concentration looks similar to others, its
> *true* concentration is much higher. This makes biological sense
> — the CFE contains all the protein from the original lysate.
> Dilution correction reveals the real picture.

</details>

# 6. Results and Interpretation

Review your outputs from Section 5 and answer the following:

**About the standard curve:**
- What was your R² value? Does it meet the ≥ 0.99 benchmark?
- Are the error bars larger at high or low concentrations?
  What might cause this in a real experiment?

**About the unknown samples:**
- Which sample has the highest **true** concentration?
  Does this match your expectation?
- How much does the dilution correction change your interpretation?
  What would you conclude if you forgot to apply it?

> 🔧 **TEMPLATE MODIFICATION POINT — Interpretation Questions**
> Replace the questions above with interpretation questions
> relevant to your own assay and experimental context.
> For example:
> - What does the slope tell you about your assay's sensitivity?
> - Do your unknown concentrations fall within the standard range?
> - What would you do if a sample absorbance was above the
>   highest standard?

> **Key takeaway:** Linear regression is not just a mathematical
> tool — it connects your instrument readings to real biological
> quantities. The same workflow applies to any assay with a linear
> response: BCA protein assay, ELISA, enzyme activity curves,
> DNA quantification, and more.

# 7. Wrap-Up

Congratulations on completing Notebook 004T!

In this notebook you:
- ✅ Loaded real experimental data from a CSV file using pandas
- ✅ Calculated means and standard deviations across replicates
- ✅ Built a publication-quality plot with error bars
- ✅ Performed linear regression with `scipy.stats.linregress`
- ✅ Evaluated fit quality using R²
- ✅ Calculated unknown concentrations and corrected for dilution

**The workflow you built here is a reusable template.**
The same steps apply to any experiment where you:
1. Measure a series of known standards
2. Fit a line to the standard curve
3. Use that line to calculate unknowns
4. Correct for sample dilution where needed

**Connections across the workshop:**

| Notebook | Key concept | How it appeared in 004T |
|---|---|---|
| 001 | Colab environment | The platform we worked in |
| 002 | Variables, loops, if statements | `slope`, `intercept`; `for` loop in 5.5 |
| 003 | DataFrames, column calculations | `pd.read_csv()`, column operations |
| 004T | Linear regression + dilution | Bringing it all together! |

**Adapting this template:**
Every 🔧 point in this notebook is a place to make it yours.
The complete list of modification points is in the header.

# Optional Challenge Questions

| Challenge | Topic | Difficulty |
|---|---|---|
| C1 | Plot unknowns on the standard curve | ⭐ |
| C2 | Calculate and plot residuals | ⭐⭐⭐ |
| C3 | Apply the workflow to a BCA assay | ⭐⭐⭐⭐ |

---

## Challenge 1 — Plot Unknowns on the Standard Curve ⭐

Add the unknown samples to the regression plot from Section 5.4.
If they fall on the best-fit line, your calculations are consistent.

1. Calculate concentrations for unknowns from `sample_abs`:
   ~~~python
   unknown_concs = [(a - intercept) / slope for a in sample_abs]
   ~~~
2. Add them as a scatter plot with a different marker and color:
   ~~~python
   plt.scatter(unknown_concs, sample_abs, marker='^',
               color='green', s=100, label='Unknown Samples')
   ~~~
3. Add sample name labels using `plt.annotate()`
4. Do all unknowns fall on the line?

---

## Challenge 2 — Residual Plot ⭐⭐⭐

R² gives a single number for fit quality but can hide patterns.
Residuals tell a richer story.

$$residual = measured\ absorbance - predicted\ absorbance$$

1. Calculate predicted absorbance:
   ~~~python
   clean_data_df['predicted'] = slope * clean_data_df[concentration_col] + intercept
   ~~~
2. Calculate residuals:
   ~~~python
   clean_data_df['residual'] = clean_data_df['average'] - clean_data_df['predicted']
   ~~~
3. Plot residuals vs. concentration with a horizontal line at y=0
4. Is there any pattern in the residuals?

---

## Challenge 3 — Apply to a BCA Assay ⭐⭐⭐⭐

The BCA (bicinchoninic acid) assay is another protein quantification
method measured at **562 nm** with a broader range (0–1000 μg/mL).

Use the data below and follow every 🔧 point to adapt the notebook:

~~~python
bca_data = {
    'Concentration_ug_per_mL': [0, 25, 125, 250, 500, 750, 1000],
    'Rep1': [0.131, 0.193, 0.379, 0.645, 1.165, 1.687, 2.199],
    'Rep2': [0.128, 0.197, 0.376, 0.650, 1.170, 1.693, 2.196],
    'Rep3': [0.130, 0.191, 0.383, 0.642, 1.163, 1.680, 2.203]
}
~~~

Unknown samples (Abs562): Sample X = 0.842, Sample Y = 1.456
Both samples were measured undiluted (dilution factor = 1).

> How does the slope compare to the Bradford assay?
> What does that tell you about the relative sensitivity of the two assays?

## Notebook Sign-Off Checklist

**General:**
* [ ] **Purpose is clear** from the header without opening code cells
* [ ] **Every significant decision** has a markdown explanation
* [ ] **Data source** is recorded in the header
* [ ] **Kernel restarted** and all cells run top-to-bottom without error

**Template-specific (before sharing with students):**
* [ ] `assay_name`, `wavelength`, `conc_units` updated in Section 3a
* [ ] `concentration_col` and `replicate_cols` match your CSV exactly
* [ ] One data entry option (A, B, or C) tested and working
* [ ] `sample_names`, `sample_abs`, and `dilution_factors`
  updated in Section 5.5
* [ ] Dilution factors confirmed — set to 1 for undiluted samples
* [ ] Interpretation questions in Section 6 updated for your context
* [ ] All outputs cleared (Runtime → Restart and clear all outputs)
  before sharing with students